# Lab 5 — Normal Models: MLE, LDA, QDA and Naive Bayes
**ENGS 3302 Pattern Recognition · Week 5** · Notes: *Week 5 — Normal Models*

| Part | What you do | How |
|---|---|---|
| **0. Warm-up** | Averages per column, and `np.cov` | Short tasks with checks |
| **1. Demo** | Fit normal curves to penguins, then compare LDA, QDA and naive Bayes | All code is given. **Watch the lab video, run each cell, change values and see what happens** |
| **2. Your turn** | Write the estimates, the distances and naive Bayes yourself | Fill in the `# TODO` cells. A check cell tells you if you are right |

**How to run:** click a cell and press **Shift + Enter**. Run the cells **from top to bottom**.
If Colab disconnects, use *Runtime → Run all*. Put `penguins.csv` (from the Moodle zip) next to this notebook, or keep the internet on.

In [ ]:
# Run this cell first. It loads the libraries, the penguin data and a helper that checks your answers.
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

def check(label, got, expected, tol=1e-3):
    """Prints a green tick if `got` is close to `expected`, otherwise a red cross."""
    try:
        ok = got is not None and np.allclose(np.asarray(got, dtype=float), np.asarray(expected, dtype=float), atol=tol)
    except Exception:
        ok = False
    print(f"✅ {label}" if ok else f"❌ {label}: expected {expected}, got {got}")

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
def load_penguins():
    """Returns the cleaned table: 342 penguins with all four measurements."""
    try:
        df = pd.read_csv("penguins.csv")
    except FileNotFoundError:
        df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
    return df.dropna(subset=FEATURES).reset_index(drop=True)   # 2 penguins have no measurements

peng = load_penguins()
print("Ready.", len(peng), "penguins")

---
## Part 0 — Warm-up
**0.1 One average per column.** `X.mean(axis=0)` gives one average for each feature (column): this is the **mean vector**.

In [ ]:
X = np.array([[1, 2], [2, 4], [3, 3]])
print("mean vector:", X.mean(axis=0))

**0.2 The covariance matrix.** `np.cov` wants one feature per **row**, so we pass `X.T`. Use `bias=True` to divide by $n$ (maximum likelihood).
Compute the covariance matrix of `X` and store it in `S`.

In [ ]:
S = None   # TODO
check("covariance matrix", S, [[0.667, 0.333], [0.333, 0.667]])

---
## Part 1 — Demo *(follow the lab video)*
### 1.1 Maximum likelihood with one feature  *(Notes §5.1)*
We fit a normal curve to the flipper length of the Gentoo penguins. We try many guesses for $\mu$ and keep the one
with the largest **log-likelihood** (the sum of the logs of the densities). The standard deviation is fixed at its estimate.

In [ ]:
x = peng[peng.species == "Gentoo"].flipper_length_mm.values
sd = x.std()                                    # np.std divides by n
guesses = np.linspace(210, 225, 301)
loglik = [norm.logpdf(x, mu, sd).sum() for mu in guesses]
best = guesses[np.argmax(loglik)]
print("best guess:", round(best, 2), "  average:", round(x.mean(), 2))
plt.plot(guesses, loglik); plt.axvline(x.mean(), color="tab:orange", ls="--")
plt.xlabel("guess for the mean (mm)"); plt.ylabel("log-likelihood"); plt.show()
print("product of the densities:", np.prod(norm.pdf(x, x.mean(), sd)))   # tiny: this is why we use logs

### 1.2 Two features: mean vector, covariance and correlation  *(Notes §5.2)*

In [ ]:
G = peng[peng.species == "Gentoo"][["flipper_length_mm", "body_mass_g"]].values
mu, S = G.mean(axis=0), np.cov(G.T, bias=True)
rho = S[0, 1] / np.sqrt(S[0, 0] * S[1, 1])
print("mean vector:", mu.round(1)); print("covariance matrix:\n", S.round(1)); print("correlation:", round(rho, 3))
plt.scatter(G[:, 0], G[:, 1], s=10); plt.xlabel("flipper length (mm)"); plt.ylabel("body mass (g)"); plt.show()

### 1.3 The threshold formula  *(Notes §5.4)*
Two classes with the same spread. We compare the formula with a search along the axis, as in Week 4.

In [ ]:
def threshold(m1, m2, s, P1):
    """Same spread s, m1 > m2: decide class 1 above the threshold."""
    return (m1 + m2) / 2 + s**2 / (m1 - m2) * np.log((1 - P1) / P1)

print("formula:", round(threshold(7, 4, 1, 0.6), 3))
xs = np.linspace(1, 10, 90001)
score1 = norm.pdf(xs, 7, 1) * 0.6; score2 = norm.pdf(xs, 4, 1) * 0.4
print("search: ", round(xs[np.argmax(score1 > score2)], 3))
print("spread 2:", round(threshold(7, 4, 2, 0.6), 3), " (the prior moves it further)")

### 1.4 LDA, QDA and naive Bayes on the penguin bills  *(Notes §5.5–5.7)*

In [ ]:
Xb = peng[["bill_length_mm", "bill_depth_mm"]].values; yb = peng.species.values
X_train, X_test, y_train, y_test = train_test_split(Xb, yb, test_size=0.3, random_state=0, stratify=yb)
models = {"LDA": LinearDiscriminantAnalysis(), "QDA": QuadraticDiscriminantAnalysis(), "naive Bayes": GaussianNB()}
fig, ax = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
gx, gy = np.meshgrid(np.linspace(30, 61, 300), np.linspace(12.5, 22, 250))
for a, (name, m) in zip(ax, models.items()):
    m.fit(X_train, y_train)
    correct = (m.predict(X_test) == y_test).sum()
    Z = np.searchsorted(m.classes_, m.predict(np.c_[gx.ravel(), gy.ravel()])).reshape(gx.shape)
    a.contourf(gx, gy, Z, alpha=0.15, levels=[-.5, .5, 1.5, 2.5]); a.contour(gx, gy, Z, levels=[.5, 1.5], colors="k")
    for sp in m.classes_:
        k = yb == sp; a.scatter(Xb[k, 0], Xb[k, 1], s=8, label=sp)
    a.set_title(f"{name}: {correct} of {len(y_test)} correct"); a.set_xlabel("bill length (mm)")
ax[0].set_ylabel("bill depth (mm)"); ax[0].legend(); plt.show()

**Try it:** in 1.4 use the four columns of `FEATURES` instead of the two bill columns (you cannot draw it, so skip the plot).
How many test penguins does each model classify correctly now?

---
## Part 2 — Your turn
### 2.1 Maximum-likelihood estimates  *(Notes §5.1)*
Write `fit_normal(x)` that returns the mean and the variance of a sample, **dividing by $n$**. Use it on the flipper length of the Adelie penguins.

In [ ]:
def fit_normal(x):
    mu = None    # TODO
    var = None   # TODO
    return mu, var

mu_a, var_a = fit_normal(peng[peng.species == "Adelie"].flipper_length_mm.values)

In [ ]:
check("Adelie mean", mu_a, 189.9536)
check("Adelie variance (divide by n)", var_a, 42.4813)

### 2.2 A covariance by hand  *(Notes, Worked example 5.2)*
For the **Chinstrap** penguins, compute the covariance of bill length and bill depth **without** `np.cov`:
the average of the products of the deviations. Then compute the correlation $\rho$.

In [ ]:
C = peng[peng.species == "Chinstrap"][["bill_length_mm", "bill_depth_mm"]].values
dev = None    # TODO: C minus its mean vector
cov12 = None  # TODO: average of dev[:, 0] * dev[:, 1]
rho = None    # TODO

In [ ]:
check("Chinstrap covariance", cov12, 2.4414)
check("Chinstrap correlation", rho, 0.6535)

### 2.3 Distances with and without the tilt  *(Notes §5.3)*
A new penguin has bill $(44, 17)$ mm. For each species compute $r^2$ **without** tilt ($z_1^2+z_2^2$) and **with** tilt
(`d @ np.linalg.inv(S) @ d`, where `d` is the difference from the mean vector and `S` the covariance matrix with `bias=True`).
Store them in two dictionaries.

In [ ]:
new = np.array([44.0, 17.0])
r2_no_tilt, r2_tilt = {}, {}
for sp in ["Adelie", "Chinstrap", "Gentoo"]:
    A = peng[peng.species == sp][["bill_length_mm", "bill_depth_mm"]].values
    # TODO: fill r2_no_tilt[sp] and r2_tilt[sp]
print(r2_no_tilt, r2_tilt)

In [ ]:
check("Chinstrap, no tilt", r2_no_tilt.get("Chinstrap"), 3.7156)
check("Chinstrap, with tilt", r2_tilt.get("Chinstrap"), 2.2917)
check("Adelie, with tilt", r2_tilt.get("Adelie"), 8.0172)

### 2.4 Naive Bayes by hand  *(Notes, Worked example 5.7)*
For the same penguin, compute for each species: prior $\times\ p(44\,|\,\omega)\times p(17\,|\,\omega)$, with one normal curve per feature
(mean and standard deviation dividing by $n$; the prior is the share of the species in all 342 penguins). Then turn the three scores into posteriors.

In [ ]:
scores = {}
for sp in ["Adelie", "Chinstrap", "Gentoo"]:
    A = peng[peng.species == sp][["bill_length_mm", "bill_depth_mm"]].values
    scores[sp] = None   # TODO
post_nb = None          # TODO: a dictionary of posteriors
print(post_nb)

In [ ]:
check("naive Bayes posterior of Adelie", None if post_nb is None else post_nb["Adelie"], 0.4074)
check("naive Bayes posterior of Chinstrap", None if post_nb is None else post_nb["Chinstrap"], 0.3132)

### Reflect (write 2–3 sentences in this cell)
In 2.3 and 2.4, naive Bayes says **Adelie**, but the distance **with** tilt says the penguin is closest to **Chinstrap**.
Which one do you trust for this penguin, and why? Look at the Chinstrap correlation from 2.2.

*Your answer:*